# 09 - Model Governance, Monitoring, and Stakeholder Reporting

**Project:** Canadian Retail Credit Risk Analytics: Portfolio Monitoring, Explainable Default Prediction, and Model Governance

This notebook consolidates the model-development, threshold-selection, and explainability outputs into governance-ready documentation. The model is treated as a **decision-support/manual-review prioritization tool**, not an automated credit-decline engine.

## Professional Governance Objectives

This notebook answers:

1. What model and threshold are approved for portfolio monitoring?
2. What evidence supports the model decision?
3. What are the key risks, controls, and monitoring limits?
4. How should technical findings be communicated to business, validation, and governance stakeholders?
5. What artifacts should be carried into the final portfolio report?

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

try:
    from credit_risk.config import GOVERNANCE_DIR, PROCESSED_DIR, TABLE_DIR, ensure_project_directories
except Exception:
    PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
    TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
    GOVERNANCE_DIR = PROJECT_ROOT / "reports" / "governance"
    def ensure_project_directories() -> None:
        for path in [PROCESSED_DIR, TABLE_DIR, GOVERNANCE_DIR]:
            path.mkdir(parents=True, exist_ok=True)

ensure_project_directories()
print(f"Project root: {PROJECT_ROOT}")
print(f"Tables:       {TABLE_DIR}")
print(f"Governance:   {GOVERNANCE_DIR}")

Project root: D:\Banking and Finance\Projects\canadian-retail-credit-risk-xai
Tables:       D:\Banking and Finance\Projects\canadian-retail-credit-risk-xai\reports\tables
Governance:   D:\Banking and Finance\Projects\canadian-retail-credit-risk-xai\reports\governance


In [2]:
from credit_risk.governance.model_governance import (
    build_control_register,
    build_feature_governance_summary,
    build_governance_readiness_gate,
    build_governance_summary,
    build_model_inventory,
    build_model_risk_limit_register,
    build_monitoring_kpi_snapshot,
    build_validation_test_summary,
    build_xai_governance_summary,
    load_governance_inputs,
    save_governance_outputs,
)

inputs = load_governance_inputs(processed_dir=PROCESSED_DIR, table_dir=TABLE_DIR)
print("Governance inputs loaded.")
print(f"Modeling rows: {len(inputs.modeling_df):,}")

Governance inputs loaded.
Modeling rows: 134,417


## 1. Model Inventory and Intended Use

The model inventory defines the approved business purpose, operating threshold, target, scope, and explicit restrictions. This distinction is important in financial services because an early-warning review model requires different controls than an automated underwriting decision model.

In [3]:
model_inventory = build_model_inventory(inputs)
model_inventory

,item,value
0,business_use,Early-warning retail credit default-risk ranki...
1,model_scope,Portfolio monitoring and decision-support; not...
2,target,Defaulter indicator
3,operational_model,xgboost_weighted_baseline
4,operating_threshold,0.56
5,threshold_objective,minimum_cost_review_rate_le_30pct
6,modeling_rows,134417
7,portfolio_default_rate,0.090413
8,feature_count,40
9,sensitive_proxy_use,Excluded from baseline model; available only f...


## 2. Validation and Test Evidence

This section compares default-threshold performance and selected operating-threshold performance. The selected operating threshold should be chosen on validation data and confirmed on the test split.

In [4]:
validation_test_summary = build_validation_test_summary(inputs)
validation_test_summary

,evaluation_view,model_name,dataset,threshold,roc_auc,pr_auc,brier_score,recall,precision,f1,balanced_accuracy,mcc,review_rate,business_cost,false_negative,false_positive,true_positive,true_negative,selected_operating_threshold
0,validation_default_0_50,xgboost_weighted_baseline,validation,0.50,0.751186,0.226251,0.202026,0.715853,0.170923,0.275957,0.685353,0.219168,0.378664,5755000.0,518,6330,1305,12010,False
1,test_default_0_50,xgboost_weighted_baseline,test,0.50,0.747839,0.214653,0.201375,0.720790,0.173534,0.279723,0.689784,0.224775,0.375539,5674000.0,509,6258,1314,12082,False
2,validation_selected_operating_threshold,xgboost_weighted_baseline,validation,0.56,0.751186,0.226251,0.202026,0.625891,0.190484,0.292077,0.680748,0.226857,0.297079,5834500.0,682,4849,1141,13491,True
3,test_selected_operating_threshold,xgboost_weighted_baseline,test,0.56,0.747839,0.214653,0.201375,0.622052,0.190877,0.292117,0.679973,0.226423,0.294649,5848500.0,689,4807,1134,13533,True


## 3. Governance Summary

This one-row summary is designed for README, portfolio report, and stakeholder communication.

In [5]:
governance_summary = build_governance_summary(inputs)
governance_summary

,operational_model,modeling_rows,portfolio_default_rate,operating_threshold,threshold_objective,test_recall,test_precision,test_review_rate,test_business_cost,primary_governance_decision
0,xgboost_weighted_baseline,134417,0.090413,0.56,minimum_cost_review_rate_le_30pct,0.622052,0.190877,0.294649,5848500.0,Use as decision-support/manual-review prioriti...


## 4. Feature Governance Summary

This section confirms whether features were used, excluded, or retained only for monitoring/governance. Leakage-prone and sensitive/proxy variables should not be used blindly in the production feature set.

In [6]:
feature_governance_summary = build_feature_governance_summary(inputs)
feature_governance_summary

,reason,feature_count
0,"Deterministic, row-level, explainable feature ...",37
1,Repayment-derived/post-origination information...,7
2,Sensitive or high-risk proxy field retained fo...,4
3,Potential socioeconomic/operational proxy; can...,3
4,High-cardinality/encrypted field is not explai...,2
5,Identifier/audit grain; useful for traceabilit...,2
6,May be valid only if known at prediction time;...,2
7,Target variable; never used as a predictor.,1


## 5. Explainability Governance Summary

Top SHAP drivers are translated into governance notes. The goal is not only to identify predictive drivers, but also to decide which drivers need drift monitoring, policy review, or stakeholder explanation.

In [7]:
xai_governance_summary = build_xai_governance_summary(inputs, top_n=12)
xai_governance_summary

,raw_feature,feature_label,mean_abs_shap,mean_shap,transformed_feature_count,positive_contribution_share,governance_note,governance_action
0,interest_rate,Interest Rate,0.628578,0.001553,1,0.562667,Pricing/risk signal: explain carefully because...,Monitor as top driver; include in periodic dri...
1,amount_missing_flag,Amount Missing Flag,0.192372,0.060985,1,0.298000,Data-quality signal: monitor for drift and avo...,Monitor as top driver; include in periodic dri...
2,amount_missing_raw_flag,Amount Missing Raw Flag,0.181842,0.058320,1,0.298000,Data-quality signal: monitor for drift and avo...,Monitor as top driver; include in periodic dri...
3,broad_data_quality_issue_count,Broad Data Quality Issue Count,0.158858,-0.032899,1,0.156000,Data-quality signal: monitor for drift and avo...,Monitor as top driver; include in periodic dri...
4,total_income_pa,Total Income Pa,0.141970,-0.020161,1,0.520667,Affordability/exposure signal: suitable for po...,Monitor as top driver; include in periodic dri...
5,core_data_quality_issue_count,Core Data Quality Issue Count,0.106240,-0.077969,1,0.689333,Data-quality signal: monitor for drift and avo...,Monitor as top driver; include in periodic dri...
6,amount,Amount,0.063021,-0.021421,1,0.525333,Affordability/exposure signal: suitable for po...,Monitor as top driver; include in periodic dri...
7,income_to_loan_buffer,Income To Loan Buffer,0.062619,0.040654,1,0.799333,Affordability/exposure signal: suitable for po...,Monitor as top driver; include in periodic dri...
8,high_interest_flag,High Interest Flag,0.061995,0.011855,1,0.322000,Pricing/risk signal: explain carefully because...,Monitor as top driver; include in periodic dri...
9,tenure_years,Tenure Years,0.061812,-0.011569,1,0.636667,Segment/product signal: monitor stability and ...,Monitor as top driver; include in periodic dri...


## 6. Model-Risk Control Register

The control register links risks to controls, evidence, owners, and monitoring cadence. This is useful for model risk management, validation review, and audit-style documentation.

In [8]:
control_register = build_control_register()
control_register

,control_area,risk,control,evidence,owner,frequency
0,Data ingestion,Many-to-many joins or duplicate keys alter tar...,Use stable borrower-record key and duplicate c...,Notebook 01/05 record-key and split outputs,Risk analytics,Each refresh
1,Data quality,Missing or inconsistent inputs distort model s...,"Track missingness, range, and logical consiste...",Notebook 02/03/08 diagnostics,Data owner,Each refresh/monthly
2,Leakage prevention,Repayment-derived variables leak future outcom...,Exclude repayment-derived variables from model...,Feature policy,Model developer,Before release
3,Sensitive/proxy governance,Sensitive or proxy variables may cause unfair ...,Exclude direct sensitive fields from baseline ...,Feature policy and fairness/proxy review,Model governance/compliance,Before release and annually
4,Model performance,Ranking performance deteriorates after portfol...,"Monitor PR-AUC, ROC-AUC, recall, precision, F1...",Notebook 06/07 outputs,Model owner,Monthly/quarterly
5,Threshold governance,Threshold creates too many reviews or too many...,Use validation-selected threshold with review-...,Notebook 07 outputs,Credit strategy,Quarterly/material change
6,Explainability,Model decisions cannot be understood by stakeh...,"Maintain SHAP, local reasons, anchor-style rul...",Notebook 08 outputs,Risk analytics,Each release
7,Monitoring,"Score, feature, or data-quality drift changes ...","Track score distribution, top-feature drift, r...",Notebook 09 monitoring plan,Model monitoring team,Monthly
8,Change management,Uncontrolled model/file changes break reproduc...,"Version model artifact, training code, feature...",Git + artifact manifest,Model owner,Each release


## 7. Monitoring KPI Snapshot

This table defines the starting point for monitoring after deployment or future data refreshes.

In [9]:
monitoring_kpis = build_monitoring_kpi_snapshot(inputs)
monitoring_kpis

,kpi,value,interpretation
0,Operational model,xgboost_weighted_baseline,Model-threshold pair selected by validation bu...
1,Portfolio default rate,9.04%,Base rate for interpreting precision and revie...
2,Threshold objective,minimum_cost_review_rate_le_30pct,Business rule used for threshold selection.
3,Operating threshold,0.560,Probability cutoff for manual-review flag.
4,Test recall,62.21%,Share of defaults captured at selected threshold.
5,Test precision,19.09%,Share of reviewed accounts that defaulted.
6,Test review rate,29.46%,Operational workload from the selected threshold.
7,Test business cost,"$5,848,500",Scenario cost from false positives and false n...
8,Top SHAP drivers,"interest_rate, amount_missing_flag, amount_mis...",Main explanation drivers to monitor for drift.


## 8. Model Risk Limit Register

The limit register gives warning and breach thresholds for score stability, feature drift, review workload, and realized-label performance.

In [10]:
risk_limits = build_model_risk_limit_register(inputs)
risk_limits

,metric,baseline,warning_limit,breach_limit,frequency,action
0,Score PSI,Training/test score distribution,> 0.10,> 0.25,Monthly,"Investigate population shift, recalibration, o..."
1,Top SHAP driver PSI,Top XAI feature distributions,> 0.10,> 0.25,Monthly,Review feature drift and data source changes.
2,Review rate,29.46%,> 34.46%,> 39.46%,Weekly/monthly,Review threshold capacity and staffing impact.
3,Recall on matured labels,62.21%,< 57.21%,< 52.21%,After labels mature,Assess missed-default concentration and refres...
4,Precision on reviewed accounts,19.09%,< 16.09%,< 14.09%,After labels mature,Review false-positive burden and threshold.
5,Critical feature missingness,Notebook 02/03 profile,+25% relative,+50% relative,Each refresh,Open data-quality incident and assess model us...


## 9. Governance Readiness Gate

This confirms whether the minimum governance artifacts exist and whether the project is ready to be summarized in the final report.

In [11]:
readiness_gate = build_governance_readiness_gate(inputs)
readiness_gate

,check,passed,note
0,modeling_dataset_available,True,Processed modelling dataset loaded.
1,threshold_recommendation_available,True,Notebook 07 operational threshold loaded.
2,test_confirmation_available,True,Selected threshold confirmed on test set.
3,xai_outputs_available,True,Notebook 08 SHAP outputs loaded.
4,control_register_created,True,Notebook 09 control register created.
5,monitoring_limits_created,True,Notebook 09 model risk limits created.
6,model_card_created,True,Model card will be saved as markdown.
7,stakeholder_brief_created,True,Stakeholder brief will be saved as markdown.


## 10. Save Governance Outputs

This saves CSV governance tables and Markdown documents:

- Model card
- Model validation summary
- Stakeholder brief
- Monitoring plan
- Control register
- Risk-limit register
- Governance output manifest

In [12]:
saved_outputs = save_governance_outputs(
    inputs,
    table_dir=TABLE_DIR,
    governance_dir=GOVERNANCE_DIR,
    project_root=PROJECT_ROOT,
)

pd.DataFrame([{"artifact": k, "path": v} for k, v in saved_outputs.items()])

,artifact,path
0,09_model_inventory.csv,D:\Banking and Finance\Projects\canadian-retai...
1,09_model_validation_test_summary.csv,D:\Banking and Finance\Projects\canadian-retai...
2,09_feature_governance_summary.csv,D:\Banking and Finance\Projects\canadian-retai...
3,09_xai_governance_summary.csv,D:\Banking and Finance\Projects\canadian-retai...
4,09_model_control_register.csv,D:\Banking and Finance\Projects\canadian-retai...
5,09_model_risk_limit_register.csv,D:\Banking and Finance\Projects\canadian-retai...
6,09_monitoring_kpi_snapshot.csv,D:\Banking and Finance\Projects\canadian-retai...
7,09_model_governance_summary.csv,D:\Banking and Finance\Projects\canadian-retai...
8,09_governance_readiness_gate.csv,D:\Banking and Finance\Projects\canadian-retai...
9,09_model_card.md,D:\Banking and Finance\Projects\canadian-retai...


## 11. Final Governance Decision

The model is acceptable for portfolio analytics and manual-review prioritization in this project context, subject to monitoring and documented limitations. It should not be positioned as an automated credit-decline engine without independent validation, calibration review, fairness testing, legal/privacy review, and production monitoring.

In [14]:
print("Notebook 09 governance pipeline completed.")

print("\nGovernance summary:")
print(governance_summary.to_string(index=False))

print("\nReadiness gate:")
print(readiness_gate.to_string(index=False))

Notebook 09 governance pipeline completed.

Governance summary:
        operational_model  modeling_rows  portfolio_default_rate  operating_threshold               threshold_objective  test_recall  test_precision  test_review_rate  test_business_cost                                                                         primary_governance_decision
xgboost_weighted_baseline         134417                0.090413                 0.56 minimum_cost_review_rate_le_30pct     0.622052        0.190877          0.294649           5848500.0 Use as decision-support/manual-review prioritization model, not as automated credit-decline engine.

Readiness gate:
                             check  passed                                         note
        modeling_dataset_available    True          Processed modelling dataset loaded.
threshold_recommendation_available    True    Notebook 07 operational threshold loaded.
       test_confirmation_available    True    Selected threshold confirmed on tes